# R8_9_helper— Homophily and edge-level brokerage metrics for bonding/bridging analysis

## Purpose

This notebook prepares the two edge-level inputs required by the refined bonding/bridging workflow in `R8_9.5_CO_bonding_bridging_split-Copy1.ipynb`.

It computes:

1. user-user homophily / similarity scores from user-level socio-demographic, behavioral, contextual, and home-distance attributes; and
2. full-graph edge-level structural metrics, including weighted edge betweenness and local constraint.

The actual bonding/bridging split, refined bridging classification, counterfactual split-graph construction, and plotting are handled in `R8_9.5_CO_bonding_bridging_split-Copy1.ipynb`. Those repetitive downstream blocks have been removed from this GitHub-ready version of `R8_9`.

## Inputs

- User-level feature table: `../Data/stop_df_perday_CO/graphs_POI_weighted/user_level_feature_table_R11.csv`
- Aligned dyadic interaction table: `../Data/stop_df_perday_CO/for_ABM/individual_interac_allusers_R11_aligned.csv`
- Census / CBG demographic table: `../Data/stop_df_perday_CO/geography_CO/CO_cbg_census_detailed.csv`
- Observed full graph pickles:
  - `PDM_Oct2021_graph_POI_weighted_R11.pkl`
  - `PDM_Nov2021_graph_POI_weighted_R11.pkl`
  - `Jan2022_remaining_pre_users_graph_POI_weighted_R11.pkl`
  - `Feb2022_remaining_pre_users_graph_POI_weighted_R11.pkl`

## Outputs

- Dyadic similarity / homophily file:
  - `../Data/stop_df_perday_CO/graphs_POI_weighted/dyad_unique_pair_similarity_R11.csv`
- Edge-level local constraint file:
  - `../Data/stop_df_perday_CO/graphs_POI_weighted/brokerage_local_constraint_edge_level_R11.csv`
- Edge-level weighted betweenness file:
  - `../Data/stop_df_perday_CO/graphs_POI_weighted/edge_betweenness_full_graph_R11.csv`
- Merged edge metric file used by `R8_9.5`:
  - `../Data/stop_df_perday_CO/graphs_POI_weighted/edge_betweenness_local_constraint_full_graph_R11.csv`

## Data access note

The real mobility data used by this workflow are proprietary Cuebiq/Spectus data and are not included in the repository. Synthetic inputs are provided only to test the pipeline structure and reproduce the expected schemas.

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from sklearn.neighbors import BallTree
import numpy as np
from scipy.sparse import lil_matrix
import json
from collections import defaultdict
import networkx as nx
import pickle
import os
import matplotlib.pyplot as plt
from shapely.geometry import Point
from shapely import wkt
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from shapely.geometry import LineString, Point
import pickle
import random
from matplotlib.lines import Line2D
import re

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
print("Current working directory:", os.getcwd())
base = os.path.join("..", "Data", "stop_df_perday_CO")
pois_dir = os.path.join(base, "POIs")
geo_dir = os.path.join(base, "geography_CO")
stops_dir = os.path.join(base, "daily_agg_to_weekly_Stops")
graph_poi_dir = os.path.join(base, "graphs", "poi-user")
os.makedirs(graph_poi_dir, exist_ok=True)
week_dir = os.path.join(graph_poi_dir, "user_connections" )
os.makedirs(week_dir, exist_ok=True)
home_dir = os.path.join(base, "home/pre_disaster")
graph_dir = os.path.join(base, "graphs_POI_weighted")
os.makedirs(graph_dir, exist_ok=True)
revision = "R11"

boot_dir = os.path.join(graph_poi_dir, "boots_user_removal", revision)
cf_out_dir = os.path.join(graph_poi_dir, "counterfactual_post", revision)

boot_dir_w = os.path.join(graph_poi_dir, "boots_user_removal_weighted_yhat", revision)
os.makedirs(boot_dir_w, exist_ok=True)

boot_dir_w_count = os.path.join(graph_poi_dir, "boots_user_removal_weighted_yhat_count", revision)
os.makedirs(boot_dir_w_count, exist_ok=True)

os.makedirs(cf_out_dir, exist_ok=True)

survivor_dir = os.path.join(graph_poi_dir, "boots_user_survivors", revision)
os.makedirs(survivor_dir, exist_ok=True)

# Loading data

In [ ]:
out_path = os.path.join(graph_dir, f"user_level_feature_table_{revision}.csv")
uf = pd.read_csv(out_path)


In [ ]:
dyad_path =  os.path.join(base, "for_ABM", f"individual_interac_allusers_{revision}_aligned.csv")
dy = pd.read_csv(dyad_path)
dy.columns

In [ ]:
from shapely import wkt
from geopy.distance import geodesic

def safe_point(x):
    if pd.isna(x):
        return None
    try:
        geom = wkt.loads(x) if isinstance(x, str) else x
        if geom.geom_type == "Point":
            return geom
    except Exception:
        pass
    return None

# Parse once
dy["cbg_centroid_i_geom"] = dy["cbg_centroid_i"].apply(safe_point)
dy["cbg_centroid_j_geom"] = dy["cbg_centroid_j"].apply(safe_point)

# Extract coordinates
dy["home_i_lon"] = dy["cbg_centroid_i_geom"].apply(lambda g: g.x if g is not None else np.nan)
dy["home_i_lat"] = dy["cbg_centroid_i_geom"].apply(lambda g: g.y if g is not None else np.nan)
dy["home_j_lon"] = dy["cbg_centroid_j_geom"].apply(lambda g: g.x if g is not None else np.nan)
dy["home_j_lat"] = dy["cbg_centroid_j_geom"].apply(lambda g: g.y if g is not None else np.nan)

# Distance
dy["distance_homes_km"] = [
    geodesic((lat_i, lon_i), (lat_j, lon_j)).km
    if pd.notna(lat_i) and pd.notna(lon_i) and pd.notna(lat_j) and pd.notna(lon_j)
    else np.nan
    for lat_i, lon_i, lat_j, lon_j in zip(
        dy["home_i_lat"], dy["home_i_lon"],
        dy["home_j_lat"], dy["home_j_lon"]
    )
]
dy["distance_homes_m"] = dy["distance_homes_km"] * 1000
dy["distance_homes_m"].head()

In [ ]:
(dy["distance_homes_m"] == 0).sum()

# user-user Homophily

In [ ]:
uf.columns

In [ ]:
race_count_cols = [
    "white_alone", "black_alone", "native_american_alone", "asian_alone",
    "pacific_islander", "some_other_race", "two_or_more_races",
    "hispanic_or_latino"
]

uf["total_population"] = uf["total_population"].replace(0, np.nan)

race_share_cols = []
for c in race_count_cols:
    new_c = c + "_share"
    uf[new_c] = uf[c] / uf["total_population"]
    race_share_cols.append(new_c)

uf[race_share_cols] = uf[race_share_cols].fillna(0)

# Optional log transform for population if not already present
uf["total_population_log"] = np.log1p(uf["total_population"])

In [ ]:
behavior_cols = [
    "H_poi_pre",
    "poi_explore_rate_pre",
    "avg_dist_km"
]

context_cols = [
    "median_age",
    "median_income_log",
    "median_rent_log",
    "fire_dist",
    "total_population_log"
]

all_needed_cols = ["user"] + behavior_cols + context_cols + race_share_cols


In [ ]:
uf2 = uf[all_needed_cols].replace([np.inf, -np.inf], np.nan).copy()

# drop rows only where behavior/context are missing
uf2 = uf2.dropna(subset=behavior_cols + context_cols).reset_index(drop=True)


In [ ]:
X_behavior = StandardScaler().fit_transform(uf2[behavior_cols])
X_context = StandardScaler().fit_transform(uf2[context_cols])
X_race = uf2[race_share_cols].fillna(0).to_numpy()

# optional combined matrix
X_all = np.hstack([X_behavior, X_context, X_race])

In [ ]:
# user -> row index
user_to_idx = dict(zip(uf2["user"], np.arange(len(uf2))))

print(f"Users in feature table after cleaning: {len(user_to_idx):,}")

In [ ]:
dy.columns

In [ ]:
dy["user_i"] = dy["user_i"].astype(str)
dy["user_j"] = dy["user_j"].astype(str)

In [ ]:
def clean_user_id(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
    except:
        pass
    return s

In [ ]:
uf2["user"] = uf2["user"].apply(clean_user_id)
dy["user_i"] = dy["user_i"].apply(clean_user_id)
dy["user_j"] = dy["user_j"].apply(clean_user_id)

In [ ]:
# Build canonical pair keys in dy

dy["pair_u1"] = np.minimum(dy["user_i"].values, dy["user_j"].values)
dy["pair_u2"] = np.maximum(dy["user_i"].values, dy["user_j"].values)

# Keep one home-distance per unique user pair
# median is safer than first in case of duplicates / noise
pair_distance_df = (
    dy.groupby(["pair_u1", "pair_u2"], as_index=False)["distance_homes_m"]
      .median()
      .rename(columns={
          "pair_u1": "user_i",
          "pair_u2": "user_j"
      })
)
pair_distance_df.head()

In [ ]:
# -----------------------------------
# Unique user-user pairs from dy
# -----------------------------------
unique_pairs = pair_distance_df[["user_i", "user_j"]].drop_duplicates().reset_index(drop=True)

user_to_idx = dict(zip(uf2["user"], np.arange(len(uf2))))

mask = (
    unique_pairs["user_i"].isin(user_to_idx) &
    unique_pairs["user_j"].isin(user_to_idx)
)

pairwise_df = unique_pairs.loc[mask].copy().reset_index(drop=True)

# merge home-distance ONCE
pairwise_df = pairwise_df.merge(
    pair_distance_df,
    on=["user_i", "user_j"],
    how="left"
)

print(f"Pairs retained after feature merge availability: {len(pairwise_df):,}")

In [ ]:
# -----------------------------------
# Map users to matrix indices
# -----------------------------------
pairwise_df["i_idx"] = pairwise_df["user_i"].map(user_to_idx)
pairwise_df["j_idx"] = pairwise_df["user_j"].map(user_to_idx)

print("Missing mapped indices:")
print(pairwise_df[["i_idx", "j_idx"]].isna().sum())

pairwise_df["i_idx"] = pairwise_df["i_idx"].astype(int)
pairwise_df["j_idx"] = pairwise_df["j_idx"].astype(int)

In [ ]:
# -----------------------------------
# Cosine similarity helper
# -----------------------------------
def row_cosine_from_matrices(X, i_idx, j_idx):
    Xi = X[i_idx]
    Xj = X[j_idx]

    numerator = np.sum(Xi * Xj, axis=1)
    denom = np.linalg.norm(Xi, axis=1) * np.linalg.norm(Xj, axis=1)

    out = np.full(len(i_idx), np.nan, dtype=float)
    valid = denom > 0
    out[valid] = numerator[valid] / denom[valid]
    return out

i_idx = pairwise_df["i_idx"].to_numpy()
j_idx = pairwise_df["j_idx"].to_numpy()

In [ ]:
# -----------------------------------
# User-level cosine similarities
# -----------------------------------
pairwise_df["sim_behavior"] = row_cosine_from_matrices(X_behavior, i_idx, j_idx)
pairwise_df["sim_context"] = row_cosine_from_matrices(X_context, i_idx, j_idx)
pairwise_df["sim_race_context"] = row_cosine_from_matrices(X_race, i_idx, j_idx)

# rescale cosine similarities from [-1, 1] to [0, 1]
for c in ["sim_behavior", "sim_context", "sim_race_context"]:
    pairwise_df[c + "_01"] = (pairwise_df[c] + 1) / 2

In [ ]:
# -----------------------------------
# Choose distance decay scale from actual distribution
# -----------------------------------
dist = pairwise_df["distance_homes_m"].replace([np.inf, -np.inf], np.nan).dropna()

print("Distance summary (meters):")
print(dist.describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))

# Recommended: median or 75th percentile
distance_scale_m = dist.quantile(0.50)

pairwise_df["sim_home_distance"] = np.exp(
    -pairwise_df["distance_homes_m"] / distance_scale_m
)

# fill missing distance similarity with median observed similarity
pairwise_df["sim_home_distance"] = pairwise_df["sim_home_distance"].fillna(
    pairwise_df["sim_home_distance"].median())

In [ ]:
# 1) similarity only
pairwise_df["sim_weighted_sim_only"] = (
    0.30 * pairwise_df["sim_behavior"] +
    0.35 * pairwise_df["sim_context"] +
    0.35 * pairwise_df["sim_race_context"]
)

# 2) similarity + distance (balanced)
pairwise_df["sim_weighted_sim_dist"] = (
    0.25 * pairwise_df["sim_behavior_01"] +
    0.25 * pairwise_df["sim_context_01"] +
    0.25 * pairwise_df["sim_race_context_01"] +
    0.25 * pairwise_df["sim_home_distance"]
)

# 3) socio-demographic + distance
pairwise_df["sim_weighted_sdm_dist"] = (
    0.25 * pairwise_df["sim_context_01"] +
    0.25 * pairwise_df["sim_race_context_01"] +
    0.50 * pairwise_df["sim_home_distance"]
)

pairwise_df = pairwise_df.drop(columns=["i_idx", "j_idx"])

In [ ]:
pairwise_df.head()

# Recompute homophily with only income and race

In [ ]:
pairwise_out_path = os.path.join(graph_dir, f"dyad_unique_pair_similarity_{revision}.csv")
pairwise_df = pd.read_csv(pairwise_out_path)
pairwise_df.columns

In [ ]:
out_path = os.path.join(graph_dir, f"user_level_feature_table_{revision}.csv")
uf = pd.read_csv(out_path)
uf.columns

In [ ]:
census_path = os.path.join(base, "geography_CO", "CO_cbg_census_detailed.csv")
cbg = pd.read_csv(census_path)
cbg.columns

In [ ]:
def clean_user_id(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
    except:
        pass
    return s

def clean_cbg(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
    except:
        pass
    return s

In [ ]:
uf = uf.copy()
cbg = cbg.copy()
pairwise_df = pairwise_df.copy()

uf["user"] = uf["user"].apply(clean_user_id)
uf["home_cbg"] = uf["pre_disaster_fips"].apply(clean_cbg)

cbg["cbg_fips"] = cbg["cbg_fips"].apply(clean_cbg)

pairwise_df["user_i"] = pairwise_df["user_i"].apply(clean_user_id)
pairwise_df["user_j"] = pairwise_df["user_j"].apply(clean_user_id)

In [ ]:
# -----------------------------
# Race columns from uf
# -----------------------------
race_count_cols = [
    "white_alone",
    "black_alone",
    "native_american_alone",
    "asian_alone",
    "pacific_islander",
    "some_other_race",
    "two_or_more_races",
    "hispanic_or_latino"
]

race_labels = [
    "white_alone",
    "black_alone",
    "native_american_alone",
    "asian_alone",
    "pacific_islander",
    "some_other_race",
    "two_or_more_races",
    "hispanic_or_latino"
]

# -----------------------------
# Income share columns from cbg
# -----------------------------
income_share_cols = [
    "hh_inc_lt_10k_share",
    "hh_inc_10k_15k_share",
    "hh_inc_15k_20k_share",
    "hh_inc_20k_25k_share",
    "hh_inc_25k_30k_share",
    "hh_inc_30k_35k_share",
    "hh_inc_35k_40k_share",
    "hh_inc_40k_45k_share",
    "hh_inc_45k_50k_share",
    "hh_inc_50k_60k_share",
    "hh_inc_60k_75k_share",
    "hh_inc_75k_100k_share",
    "hh_inc_100k_125k_share",
    "hh_inc_125k_150k_share",
    "hh_inc_150k_200k_share",
    "hh_inc_200k_plus_share"
]

income_labels = [
    "hh_inc_lt_10k",
    "hh_inc_10k_15k",
    "hh_inc_15k_20k",
    "hh_inc_20k_25k",
    "hh_inc_25k_30k",
    "hh_inc_30k_35k",
    "hh_inc_35k_40k",
    "hh_inc_40k_45k",
    "hh_inc_45k_50k",
    "hh_inc_50k_60k",
    "hh_inc_60k_75k",
    "hh_inc_75k_100k",
    "hh_inc_100k_125k",
    "hh_inc_125k_150k",
    "hh_inc_150k_200k",
    "hh_inc_200k_plus"
]

In [ ]:
uf["total_population"] = uf["total_population"].replace(0, np.nan)

race_share_cols = []
for c in race_count_cols:
    share_col = c + "_share"
    uf[share_col] = uf[c] / uf["total_population"]
    race_share_cols.append(share_col)

uf[race_share_cols] = uf[race_share_cols].fillna(0)


In [ ]:
# -----------------------------
# Merge income shares into uf
# -----------------------------
cbg_income = cbg[["cbg_fips"] + income_share_cols].drop_duplicates("cbg_fips").copy()

uf = uf.merge(
    cbg_income,
    left_on="home_cbg",
    right_on="cbg_fips",
    how="left"
)

# optional cleanup
uf = uf.drop(columns=["cbg_fips"], errors="ignore")

In [ ]:
# -----------------------------
# User attribute base table
# -----------------------------
user_attr = uf[["user", "home_cbg"] + race_share_cols + income_share_cols].copy()

# one row per user
user_attr = user_attr.drop_duplicates(subset=["user"]).reset_index(drop=True)

# drop users with missing home CBG
user_attr = user_attr.dropna(subset=["user", "home_cbg"]).reset_index(drop=True)

In [ ]:
# -----------------------------
# Safe normalization
# -----------------------------
def normalize_rows(arr):
    arr = np.asarray(arr, dtype=float)
    arr = np.where(np.isfinite(arr), arr, 0.0)
    row_sums = arr.sum(axis=1, keepdims=True)
    valid = row_sums.squeeze() > 0
    out = np.zeros_like(arr, dtype=float)
    out[valid] = arr[valid] / row_sums[valid]
    return out, valid

race_probs, valid_race = normalize_rows(user_attr[race_share_cols].to_numpy())
income_probs, valid_income = normalize_rows(user_attr[income_share_cols].to_numpy())

user_attr["valid_race_probs"] = valid_race
user_attr["valid_income_probs"] = valid_income

print("Users with valid race probs:", valid_race.sum())
print("Users with valid income probs:", valid_income.sum())
print("Users valid for both:", (valid_race & valid_income).sum())

In [ ]:
# -----------------------------
# Sampling helper
# -----------------------------
def sample_from_prob_matrix(prob_matrix, labels, seed=42):
    rng = np.random.default_rng(seed)
    cum_probs = np.cumsum(prob_matrix, axis=1)
    draws = rng.random((prob_matrix.shape[0], 1))
    sampled_idx = (draws <= cum_probs).argmax(axis=1)
    return pd.Categorical(np.array(labels)[sampled_idx], categories=labels)

# -----------------------------
# Sample attributes
# -----------------------------
user_attr["race_group"] = pd.NA
user_attr["income_group"] = pd.NA

mask_race = user_attr["valid_race_probs"].to_numpy()
mask_income = user_attr["valid_income_probs"].to_numpy()

user_attr.loc[mask_race, "race_group"] = sample_from_prob_matrix(
    race_probs[mask_race],
    race_labels,
    seed=42
).astype(str)

user_attr.loc[mask_income, "income_group"] = sample_from_prob_matrix(
    income_probs[mask_income],
    income_labels,
    seed=4242
).astype(str)

# ordered income-bin index for similarity
income_to_idx = {lab: i for i, lab in enumerate(income_labels)}
user_attr["income_idx"] = user_attr["income_group"].map(income_to_idx)

user_attr.head()

In [ ]:
# -----------------------------
# Add sampled attributes to uf
# -----------------------------
uf = uf.merge(
    user_attr[["user", "race_group", "income_group", "income_idx"]],
    on="user",
    how="left"
)

uf.head()

In [ ]:
# -----------------------------
# Attach user_i / user_j attributes to dyads
# -----------------------------
user_i_attrs = user_attr[["user", "race_group", "income_group", "income_idx"]].rename(columns={
    "user": "user_i",
    "race_group": "race_i",
    "income_group": "income_i",
    "income_idx": "income_idx_i"
})

user_j_attrs = user_attr[["user", "race_group", "income_group", "income_idx"]].rename(columns={
    "user": "user_j",
    "race_group": "race_j",
    "income_group": "income_j",
    "income_idx": "income_idx_j"
})

pairwise_df = pairwise_df.merge(user_i_attrs, on="user_i", how="left")
pairwise_df = pairwise_df.merge(user_j_attrs, on="user_j", how="left")

In [ ]:
# -----------------------------
# Compute dyadic similarity
# -----------------------------
pairwise_df["race_sim"] = np.where(
    pairwise_df["race_i"].notna() & pairwise_df["race_j"].notna(),
    (pairwise_df["race_i"] == pairwise_df["race_j"]).astype(float),
    np.nan
)

n_income_bins = len(income_labels)

pairwise_df["income_sim"] = np.where(
    pairwise_df["income_idx_i"].notna() & pairwise_df["income_idx_j"].notna(),
    1 - (
        np.abs(pairwise_df["income_idx_i"] - pairwise_df["income_idx_j"]) / (n_income_bins - 1)
    ),
    np.nan
)

pairwise_df["race_inc_sim"] = 0.5 * pairwise_df["race_sim"] + 0.5 * pairwise_df["income_sim"]

In [ ]:
pairwise_df.columns

In [ ]:
pairwise_out_path = os.path.join(graph_dir, f"dyad_unique_pair_similarity_{revision}.csv")
pairwise_df.to_csv(pairwise_out_path, index=False)

print("✅ Saved pairwise similarity file:", pairwise_out_path)

In [ ]:
sim_col = "race_sim_mean"
thr = pairwise_df[sim_col].dropna().median()

plt.figure(figsize=(8,5))
plt.hist(pairwise_df[sim_col].dropna(), bins=50)
plt.axvline(thr, linestyle="--", label=f"median = {thr:.3f}")
plt.xlabel(sim_col)
plt.ylabel("Count")
plt.legend()
plt.show()

# Homophily one function (race and income ) bootstrapped 100 times

In [ ]:
# ============================================================
# SETTINGS
# ============================================================
revision = "R11"
n_draws = 100
base_seed = 42

census_path = os.path.join(base, "geography_CO", "CO_cbg_census_detailed.csv")
cbg = pd.read_csv(census_path)

# ============================================================
# HELPERS
# ============================================================
def clean_user_id(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
    except:
        pass
    return s

def clean_cbg(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
    except:
        pass
    return s

def normalize_rows(arr):
    """
    Row-normalize probability arrays safely.
    Returns:
        out   : normalized array
        valid : boolean mask for rows with positive row sums
    """
    arr = np.asarray(arr, dtype=float)
    arr = np.where(np.isfinite(arr), arr, 0.0)
    row_sums = arr.sum(axis=1, keepdims=True)
    valid = row_sums.squeeze() > 0
    out = np.zeros_like(arr, dtype=float)
    out[valid] = arr[valid] / row_sums[valid]
    return out, valid

def sample_from_prob_matrix(prob_matrix, labels, seed=42):
    """
    Sample one category per row from a probability matrix.
    """
    rng = np.random.default_rng(seed)
    cum_probs = np.cumsum(prob_matrix, axis=1)
    draws = rng.random((prob_matrix.shape[0], 1))
    sampled_idx = (draws <= cum_probs).argmax(axis=1)
    return np.array(labels, dtype=object)[sampled_idx]

# ============================================================
# CLEAN INPUT TABLES
# ============================================================
uf = uf.copy()
dy = dy.copy()
cbg = cbg.copy()

uf["user"] = uf["user"].apply(clean_user_id)
uf["home_cbg"] = uf["pre_disaster_fips"].apply(clean_cbg)

cbg["cbg_fips"] = cbg["cbg_fips"].apply(clean_cbg)

dy["user_i"] = dy["user_i"].apply(clean_user_id)
dy["user_j"] = dy["user_j"].apply(clean_user_id)

# ============================================================
# BUILD UNIQUE USER-USER PAIRS + HOME DISTANCE
# ============================================================
# canonical pair keys
dy["pair_u1"] = np.minimum(dy["user_i"].values, dy["user_j"].values)
dy["pair_u2"] = np.maximum(dy["user_i"].values, dy["user_j"].values)

# one home-distance per pair
pair_distance_df = (
    dy.groupby(["pair_u1", "pair_u2"], as_index=False)["distance_homes_m"]
      .median()
      .rename(columns={"pair_u1": "user_i", "pair_u2": "user_j"})
)

unique_pairs = pair_distance_df[["user_i", "user_j"]].drop_duplicates().reset_index(drop=True)

print("Unique user-user pairs in dy:", len(unique_pairs))

# ============================================================
# PREPARE USER-LEVEL RACE + INCOME DISTRIBUTIONS
# ============================================================
# ---- race from uf/home-CBG context already present in uf ----
race_count_cols = [
    "white_alone",
    "black_alone",
    "native_american_alone",
    "asian_alone",
    "pacific_islander",
    "some_other_race",
    "two_or_more_races",
    "hispanic_or_latino"
]

race_labels = [
    "white_alone",
    "black_alone",
    "native_american_alone",
    "asian_alone",
    "pacific_islander",
    "some_other_race",
    "two_or_more_races",
    "hispanic_or_latino"
]

# ---- income shares from detailed ACS cbg table ----
income_share_cols = [
    "hh_inc_lt_10k_share",
    "hh_inc_10k_15k_share",
    "hh_inc_15k_20k_share",
    "hh_inc_20k_25k_share",
    "hh_inc_25k_30k_share",
    "hh_inc_30k_35k_share",
    "hh_inc_35k_40k_share",
    "hh_inc_40k_45k_share",
    "hh_inc_45k_50k_share",
    "hh_inc_50k_60k_share",
    "hh_inc_60k_75k_share",
    "hh_inc_75k_100k_share",
    "hh_inc_100k_125k_share",
    "hh_inc_125k_150k_share",
    "hh_inc_150k_200k_share",
    "hh_inc_200k_plus_share"
]

income_labels = [
    "hh_inc_lt_10k",
    "hh_inc_10k_15k",
    "hh_inc_15k_20k",
    "hh_inc_20k_25k",
    "hh_inc_25k_30k",
    "hh_inc_30k_35k",
    "hh_inc_35k_40k",
    "hh_inc_40k_45k",
    "hh_inc_45k_50k",
    "hh_inc_50k_60k",
    "hh_inc_60k_75k",
    "hh_inc_75k_100k",
    "hh_inc_100k_125k",
    "hh_inc_125k_150k",
    "hh_inc_150k_200k",
    "hh_inc_200k_plus"
]

# race shares from uf
uf["total_population"] = uf["total_population"].replace(0, np.nan)

race_share_cols = []
for c in race_count_cols:
    share_col = c + "_share"
    uf[share_col] = uf[c] / uf["total_population"]
    race_share_cols.append(share_col)

uf[race_share_cols] = uf[race_share_cols].fillna(0)

# merge income shares from cbg onto uf
cbg_income = cbg[["cbg_fips"] + income_share_cols].drop_duplicates("cbg_fips").copy()

uf = uf.merge(
    cbg_income,
    left_on="home_cbg",
    right_on="cbg_fips",
    how="left"
)

uf = uf.drop(columns=["cbg_fips"], errors="ignore")

# one user row for sampling
user_attr = uf[["user", "home_cbg"] + race_share_cols + income_share_cols].copy()
user_attr = user_attr.drop_duplicates(subset=["user"]).reset_index(drop=True)
user_attr = user_attr.dropna(subset=["user", "home_cbg"]).reset_index(drop=True)

# normalize probabilities
race_probs, valid_race = normalize_rows(user_attr[race_share_cols].to_numpy())
income_probs, valid_income = normalize_rows(user_attr[income_share_cols].to_numpy())

user_attr["valid_race_probs"] = valid_race
user_attr["valid_income_probs"] = valid_income
user_attr["valid_both"] = valid_race & valid_income

print("Users in user_attr:", len(user_attr))
print("Users with valid race probs:", int(valid_race.sum()))
print("Users with valid income probs:", int(valid_income.sum()))
print("Users valid for both:", int((valid_race & valid_income).sum()))

# keep only users valid for both
user_attr = user_attr[user_attr["valid_both"]].reset_index(drop=True)

# keep only dyads where both users are in user_attr
valid_users = set(user_attr["user"])
pairwise_df = unique_pairs[
    unique_pairs["user_i"].isin(valid_users) &
    unique_pairs["user_j"].isin(valid_users)
].copy()

pairwise_df = pairwise_df.merge(
    pair_distance_df,
    on=["user_i", "user_j"],
    how="left"
)

print("Pairs retained after valid user filtering:", len(pairwise_df))

# map user -> row index in user_attr
user_to_idx = dict(zip(user_attr["user"], np.arange(len(user_attr))))

pairwise_df["i_idx"] = pairwise_df["user_i"].map(user_to_idx)
pairwise_df["j_idx"] = pairwise_df["user_j"].map(user_to_idx)

assert pairwise_df["i_idx"].notna().all()
assert pairwise_df["j_idx"].notna().all()

pairwise_df["i_idx"] = pairwise_df["i_idx"].astype(int)
pairwise_df["j_idx"] = pairwise_df["j_idx"].astype(int)

i_idx = pairwise_df["i_idx"].to_numpy()
j_idx = pairwise_df["j_idx"].to_numpy()

income_to_idx = {lab: i for i, lab in enumerate(income_labels)}
n_income_bins = len(income_labels)

# ============================================================
# MULTIPLE IMPUTATIONS: 30 DRAWS PER USER
# ============================================================
user_draw_rows = []
pair_draw_rows = []

race_probs_valid = race_probs[user_attr.index]
income_probs_valid = income_probs[user_attr.index]

for draw in range(n_draws):
    seed_race = base_seed + draw
    seed_income = base_seed + 10_000 + draw

    sampled_race = sample_from_prob_matrix(
        race_probs_valid,
        race_labels,
        seed=seed_race
    )

    sampled_income = sample_from_prob_matrix(
        income_probs_valid,
        income_labels,
        seed=seed_income
    )

    sampled_income_idx = pd.Series(sampled_income).map(income_to_idx).to_numpy()

    # ---------- store user-level draw table ----------
    user_draw_df = pd.DataFrame({
        "user": user_attr["user"].values,
        "home_cbg": user_attr["home_cbg"].values,
        "draw": draw,
        "race_group": sampled_race,
        "income_group": sampled_income,
        "income_idx": sampled_income_idx
    })
    user_draw_rows.append(user_draw_df)

    # ---------- compute pairwise similarity for this draw ----------
    race_i = sampled_race[i_idx]
    race_j = sampled_race[j_idx]

    income_i = sampled_income[i_idx]
    income_j = sampled_income[j_idx]

    income_idx_i = sampled_income_idx[i_idx]
    income_idx_j = sampled_income_idx[j_idx]

    race_sim = (race_i == race_j).astype(float)

    income_sim = 1 - (
        np.abs(income_idx_i - income_idx_j) / (n_income_bins - 1)
    )

    race_inc_sim = 0.5 * race_sim + 0.5 * income_sim

    pair_draw_df = pd.DataFrame({
        "user_i": pairwise_df["user_i"].values,
        "user_j": pairwise_df["user_j"].values,
        "distance_homes_m": pairwise_df["distance_homes_m"].values,
        "draw": draw,
        "race_i": race_i,
        "race_j": race_j,
        "income_i": income_i,
        "income_j": income_j,
        "income_idx_i": income_idx_i,
        "income_idx_j": income_idx_j,
        "race_sim": race_sim,
        "income_sim": income_sim,
        "race_inc_sim": race_inc_sim
    })
    pair_draw_rows.append(pair_draw_df)

# concatenate
user_draws_df = pd.concat(user_draw_rows, ignore_index=True)
pair_draws_df = pd.concat(pair_draw_rows, ignore_index=True)

print("User-draw table shape:", user_draws_df.shape)
print("Pair-draw table shape:", pair_draws_df.shape)

# ============================================================
# PAIRWISE SUMMARY ACROSS 100 DRAWS
# ============================================================
pairwise_summary_df = (
    pair_draws_df
    .groupby(["user_i", "user_j"], as_index=False)
    .agg(
        distance_homes_m=("distance_homes_m", "first"),
        race_inc_sim_mean=("race_inc_sim", "mean"),
        race_inc_sim_std=("race_inc_sim", "std"),
        race_inc_sim_median=("race_inc_sim", "median"),
        race_inc_sim_min=("race_inc_sim", "min"),
        race_inc_sim_max=("race_inc_sim", "max"),
        race_sim_mean=("race_sim", "mean"),
        income_sim_mean=("income_sim", "mean")
    )
)

print(pairwise_summary_df.head())

# optional: coefficient of variation only where mean > 0
pairwise_summary_df["race_inc_sim_cv"] = np.where(
    pairwise_summary_df["race_inc_sim_mean"] > 0,
    pairwise_summary_df["race_inc_sim_std"] / pairwise_summary_df["race_inc_sim_mean"],
    np.nan
)



In [ ]:
pairwise_summary_df.head()

In [ ]:
# ---------------------------------------------------
# Merge 100-draw summary back into existing pairwise_df
# ---------------------------------------------------

def clean_user_id(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
    except:
        pass
    return s

pairwise_df = pairwise_df.copy()
pairwise_summary_df = pairwise_summary_df.copy()

# make sure IDs align
pairwise_df["user_i"] = pairwise_df["user_i"].apply(clean_user_id)
pairwise_df["user_j"] = pairwise_df["user_j"].apply(clean_user_id)

pairwise_summary_df["user_i"] = pairwise_summary_df["user_i"].apply(clean_user_id)
pairwise_summary_df["user_j"] = pairwise_summary_df["user_j"].apply(clean_user_id)

# optional sanity check
print("Rows in original pairwise_df:", len(pairwise_df))
print("Rows in pairwise_summary_df:", len(pairwise_summary_df))

# columns to merge in
summary_cols = [
    "user_i", "user_j",
    "race_inc_sim_mean",
    "race_inc_sim_std",
    "race_inc_sim_median",
    "race_inc_sim_min",
    "race_inc_sim_max",
    "race_sim_mean",
    "income_sim_mean",
    "race_inc_sim_cv"
]

pairwise_df = pairwise_df.merge(
    pairwise_summary_df[summary_cols],
    on=["user_i", "user_j"],
    how="left"
)

# optional checks
print("\nNew columns added:")
print([
    c for c in [
        "race_inc_sim_mean",
        "race_inc_sim_std",
        "race_inc_sim_median",
        "race_inc_sim_min",
        "race_inc_sim_max",
        "race_sim_mean",
        "income_sim_mean",
        "race_inc_sim_cv"
    ] if c in pairwise_df.columns
])

print("\nMissing values after merge:")
print(pairwise_df[
    [
        "race_inc_sim_mean",
        "race_inc_sim_std",
        "race_inc_sim_median",
        "race_inc_sim_min",
        "race_inc_sim_max",
        "race_sim_mean",
        "income_sim_mean",
        "race_inc_sim_cv"
    ]
].isna().sum())

# ---------------------------------------------------
# Save back to same file
# ---------------------------------------------------
pairwise_out_path = os.path.join(graph_dir, f"dyad_unique_pair_similarity_{revision}.csv")
pairwise_df.to_csv(pairwise_out_path, index=False)

print("✅ Saved pairwise similarity file with additional columns:", pairwise_out_path)
print("\nFinal columns:")
print(pairwise_df.columns)

In [ ]:
sim_col = "race_inc_sim_mean"
thr = pairwise_df[sim_col].dropna().median()

plt.figure(figsize=(5,3))
plt.hist(pairwise_df[sim_col].dropna(), bins=50)
plt.axvline(thr, linestyle="--", label=f"median = {thr:.3f}")
plt.xlabel("Similarity (Homophily)")
plt.ylabel("Count")
plt.legend()
plt.show()

# Computing brokerage and edge betweenness for full graph - Pre and Post

In [ ]:
revisions = ["R11"]


def load_nx_graph(pkl_path):
    with open(pkl_path, "rb") as f:
        G = pickle.load(f)
    if not isinstance(G, (nx.Graph, nx.DiGraph, nx.MultiGraph, nx.MultiDiGraph)):
        raise TypeError(f"Loaded object is not a NetworkX graph: {type(G)} from {pkl_path}")
    return G

def ensure_str_nodes(G):
    if any(not isinstance(n, str) for n in G.nodes()):
        return nx.relabel_nodes(G, {n: str(n) for n in G.nodes()}, copy=True)
    return G

def build_full_graph_files_for_revision(graph_dir, revision):
    return {
        "Oct2021": os.path.join(graph_dir, f"PDM_Oct2021_graph_POI_weighted_{revision}.pkl"),
        "Nov2021": os.path.join(graph_dir, f"PDM_Nov2021_graph_POI_weighted_{revision}.pkl"),
        "Jan2022": os.path.join(graph_dir, f"Jan2022_remaining_pre_users_graph_POI_weighted_{revision}.pkl"),
        "Feb2022": os.path.join(graph_dir, f"Feb2022_remaining_pre_users_graph_POI_weighted_{revision}.pkl"),
    }

## Brokerage

### Edge level

In [ ]:
# ---------------------------------------------------
# EDGE-LEVEL local constraint
# ---------------------------------------------------
def compute_local_constraint_edge_df(G, weight="weight"):
    """
    Edge-level brokerage table using local constraint.

    For each undirected edge (u, v), compute:
        - local_constraint_uv = local constraint of u with respect to v
        - local_constraint_vu = local constraint of v with respect to u
        - local_constraint_mean = average of the two
        - local_constraint_min  = min of the two
        - edge_brokerage_proxy_mean = 1 - local_constraint_mean
        - edge_brokerage_proxy_min  = 1 - local_constraint_min

    Lower local constraint = more brokerage-like / less redundant tie.
    """
    G = ensure_str_nodes(G)

    rows = []

    for u, v, data in G.edges(data=True):
        lc_uv = nx.local_constraint(G, u, v, weight=weight)
        lc_vu = nx.local_constraint(G, v, u, weight=weight)

        lc_mean = np.nanmean([lc_uv, lc_vu])
        lc_min = np.nanmin([lc_uv, lc_vu])

        rows.append({
            "source": u,
            "target": v,
            "weight": data.get(weight, np.nan),

            "local_constraint_uv": lc_uv,
            "local_constraint_vu": lc_vu,
            "local_constraint_mean": lc_mean,
            "local_constraint_min": lc_min,

            "edge_brokerage_proxy_mean": 1 - lc_mean if pd.notna(lc_mean) else np.nan,
            "edge_brokerage_proxy_min": 1 - lc_min if pd.notna(lc_min) else np.nan
        })

    out = pd.DataFrame(rows)
    return out

def summarize_local_constraint(edge_df):
    valid_mean = edge_df["local_constraint_mean"].dropna()
    valid_min = edge_df["local_constraint_min"].dropna()

    if len(valid_mean) == 0:
        return {
            "edges": len(edge_df),
            "edges_with_local_constraint_mean": 0,
            "mean_local_constraint_mean": np.nan,
            "median_local_constraint_mean": np.nan,
            "std_local_constraint_mean": np.nan,
            "mean_edge_brokerage_proxy_mean": np.nan,
            "median_edge_brokerage_proxy_mean": np.nan,

            "edges_with_local_constraint_min": 0,
            "mean_local_constraint_min": np.nan,
            "median_local_constraint_min": np.nan,
            "std_local_constraint_min": np.nan,
            "mean_edge_brokerage_proxy_min": np.nan,
            "median_edge_brokerage_proxy_min": np.nan,
        }

    return {
        "edges": len(edge_df),

        "edges_with_local_constraint_mean": int(valid_mean.notna().sum()),
        "mean_local_constraint_mean": float(valid_mean.mean()),
        "median_local_constraint_mean": float(valid_mean.median()),
        "std_local_constraint_mean": float(valid_mean.std()),
        "mean_edge_brokerage_proxy_mean": float((1 - valid_mean).mean()),
        "median_edge_brokerage_proxy_mean": float((1 - valid_mean).median()),

        "edges_with_local_constraint_min": int(valid_min.notna().sum()),
        "mean_local_constraint_min": float(valid_min.mean()),
        "median_local_constraint_min": float(valid_min.median()),
        "std_local_constraint_min": float(valid_min.std()),
        "mean_edge_brokerage_proxy_min": float((1 - valid_min).mean()),
        "median_edge_brokerage_proxy_min": float((1 - valid_min).median()),
    }

In [ ]:
# ---------------------------------------------------
# run
# ---------------------------------------------------
revisions = ["R11"]

for revision in revisions:
    graph_files = build_full_graph_files_for_revision(graph_dir, revision)

    edge_rows = []
    network_rows = []

    for month, graph_path in graph_files.items():
        if not os.path.exists(graph_path):
            print(f"❌ Missing: {graph_path}")
            continue

        print(f"📦 Processing {month}")

        G = load_nx_graph(graph_path)
        G = ensure_str_nodes(G)

        phase = "pre" if month in ["Oct2021", "Nov2021"] else "post"

        edge_df = compute_local_constraint_edge_df(G, weight="weight")
        edge_df["revision"] = revision
        edge_df["month"] = month
        edge_df["phase"] = phase

        edge_rows.append(edge_df)

        net = summarize_local_constraint(edge_df)
        net.update({
            "revision": revision,
            "month": month,
            "phase": phase
        })
        network_rows.append(net)

    brokerage_edge_df = pd.concat(edge_rows, ignore_index=True)
    brokerage_network_df = pd.DataFrame(network_rows)

# preview
brokerage_edge_df.head()

In [ ]:
brokerage_edge_path = os.path.join(
    graph_dir, f"brokerage_local_constraint_edge_level_{revision}.csv"
)
brokerage_network_path = os.path.join(
    graph_dir, f"brokerage_local_constraint_network_level_{revision}.csv"
)

brokerage_edge_df.to_csv(brokerage_edge_path, index=False)
brokerage_network_df.to_csv(brokerage_network_path, index=False)

print("✅ Saved edge-level:", brokerage_edge_path)
print("✅ Saved network-level:", brokerage_network_path)

## Edge Betweenness

In [ ]:
def add_inverse_distance_attribute_undirected(G, weight="weight", dist="dist", eps=1e-12):
    """
    Convert tie strength to path distance for shortest-path based metrics.
    Larger strength -> smaller distance.
    """
    G = G.copy()

    for u, v, data in G.edges(data=True):
        w = data.get(weight, np.nan)

        if pd.isna(w) or w <= 0:
            data[dist] = np.inf
        else:
            data[dist] = 1.0 / max(w, eps)

    return G

def compute_edge_betweenness_df(G, weight_for_paths="dist"):
    """
    Compute edge betweenness centrality for all edges.
    weight_for_paths should be:
      - None for unweighted
      - 'dist' for weighted shortest paths using inverse tie strength
    """
    G = ensure_str_nodes(G)

    ebc = nx.edge_betweenness_centrality(G, weight=weight_for_paths, normalized=True)

    rows = []
    for (u, v), val in ebc.items():
        data = G.get_edge_data(u, v, default={})
        rows.append({
            "source": u,
            "target": v,
            "weight": data.get("weight", np.nan),
            "dist": data.get("dist", np.nan),
            "edge_betweenness": val
        })

    out = pd.DataFrame(rows)
    return out

In [ ]:
for revision in revisions:
    graph_files = build_full_graph_files_for_revision(graph_dir, revision)

    edge_rows = []

    for month, graph_path in graph_files.items():
        if not os.path.exists(graph_path):
            print(f"❌ Missing: {graph_path}")
            continue

        print(f"📦 Processing edge betweenness: {month}")

        G = load_nx_graph(graph_path)
        G = ensure_str_nodes(G)

        phase = "pre" if month in ["Oct2021", "Nov2021"] else "post"

        # convert strength to distance for shortest-path metric
        G = add_inverse_distance_attribute_undirected(G, weight="weight", dist="dist")

        edge_df = compute_edge_betweenness_df(G, weight_for_paths="dist")
        edge_df["revision"] = revision
        edge_df["month"] = month
        edge_df["phase"] = phase

        edge_rows.append(edge_df)

    edge_betweenness_df = pd.concat(edge_rows, ignore_index=True)

edge_betweenness_df.head()

In [ ]:
edge_betweenness_path = os.path.join(
    graph_dir,
    f"edge_betweenness_full_graph_{revision}.csv"
)

edge_betweenness_df.to_csv(edge_betweenness_path, index=False)

print("✅ Saved edge betweenness:", edge_betweenness_path)

## Merge brokerage onto the edge betweenness dataframe

In [ ]:
revision = "R11"

local_constraint_edge_path = os.path.join(
    graph_dir, f"brokerage_local_constraint_edge_level_{revision}.csv"
)

edge_betweenness_path = os.path.join(
    graph_dir, f"edge_betweenness_full_graph_{revision}.csv"
)

local_constraint_edge_df = pd.read_csv(local_constraint_edge_path)
edge_betweenness_df = pd.read_csv(edge_betweenness_path)

# make sure IDs are strings
for df in [local_constraint_edge_df, edge_betweenness_df]:
    df["source"] = df["source"].astype(str)
    df["target"] = df["target"].astype(str)
    df["revision"] = df["revision"].astype(str)
    df["month"] = df["month"].astype(str)
    df["phase"] = df["phase"].astype(str)


In [ ]:
# optional: if edge ordering may differ across files, canonicalize undirected edges
def canonicalize_edge_order(df):
    df = df.copy()
    s = df["source"].astype(str)
    t = df["target"].astype(str)
    df["source"] = np.where(s <= t, s, t)
    df["target"] = np.where(s <= t, t, s)
    return df

local_constraint_edge_df = canonicalize_edge_order(local_constraint_edge_df)
edge_betweenness_df = canonicalize_edge_order(edge_betweenness_df)

# keep only needed local-constraint columns
local_cols = [
    "source", "target", "revision", "month", "phase",
    "local_constraint_uv", "local_constraint_vu",
    "local_constraint_mean", "local_constraint_min",
    "edge_brokerage_proxy_mean", "edge_brokerage_proxy_min"
]

edge_with_metrics = edge_betweenness_df.merge(
    local_constraint_edge_df[local_cols],
    on=["source", "target", "revision", "month", "phase"],
    how="left"
)



In [ ]:
edge_with_metrics.head()

In [ ]:
out_path = os.path.join(
    graph_dir,
    f"edge_betweenness_local_constraint_full_graph_{revision}.csv"
)

edge_with_metrics.to_csv(out_path, index=False)

print("✅ Saved:", out_path)